# THIWASCO Leak Detection - Fast Model Training

**Simplified version for quick results with enhanced features**

## Focus:
- Train 2 best models (Random Forest + XGBoost)
- Test enhanced features for moderate/instant leak detection
- Save models for real-time system

In [ ]:
# Quick imports
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler

print("Training setup ready!")

In [ ]:
# Load data and prepare quickly
data_dir = Path().absolute().parent / 'data'
features_path = data_dir / 'processed' / 'engineered_features.csv'

if features_path.exists():
    df = pd.read_csv(features_path)
    print(f"Loaded {len(df)} samples with {len(df.columns)} features")
    
    # Enhanced features check
    enhanced_features = [
        'flow_burst_magnitude', 'pressure_drop_magnitude',
        'instant_flow_jump', 'instant_pressure_drop',
        'moderate_flow_sustained', 'moderate_pressure_sustained',
        'high_demand_flow_ratio', 'flow_pattern_consistency',
        'pressure_pattern_consistency'
    ]
    
    available_enhanced = [f for f in enhanced_features if f in df.columns]
    print(f"Enhanced features: {len(available_enhanced)}/9")
    
    # Prepare features and target
    feature_cols = [col for col in df.columns if col not in ['scenario', 'time_index', 'scenario_has_leak', 'leak_type', 'leak_active_at_timestep']]
    X = df[feature_cols].fillna(df[feature_cols].median())  # Quick NaN fix
    y = df['leak_type']
    
    # Binary classification (none vs leak)
    y_binary = (y != 'none').astype(int)
    
    print(f"Binary distribution: {y_binary.value_counts().to_dict()}")
    print(f"Target distribution: {dict(y.value_counts())}")
else:
    print("Feature file not found!")
    df = None

In [ ]:
# Split data for both tasks
if df is not None:
    # Binary classification split
    X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
        X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
    )
    
    # Multi-class split
    X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale for XGBoost
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bin)
    X_test_scaled = scaler.transform(X_test_bin)
    
    print(f"Binary train: {X_train_bin.shape}, Test: {X_test_bin.shape}")
    print(f"Multi-class train: {X_train_multi.shape}, Test: {X_test_multi.shape}")
    print(f"Leak ratio - Train: {y_train_bin.mean():.3f}, Test: {y_test_bin.mean():.3f}")
else:
    print("No data available!")

## Binary Classification (Leak vs No Leak)

In [ ]:
# Train 2 best binary models
if df is not None:
    print("=== BINARY CLASSIFICATION ===")
    
    # Random Forest (fast and reliable)
    rf_binary = RandomForestClassifier(
        n_estimators=200,  # Reduced for speed
        max_depth=15,
        random_state=42,
        n_jobs=-1  # Use all cores
    )
    
    rf_binary.fit(X_train_bin, y_train_bin)
    rf_pred = rf_binary.predict(X_test_bin)
    rf_f1 = f1_score(y_test_bin, rf_pred)
    rf_acc = accuracy_score(y_test_bin, rf_pred)
    
    print(f"Random Forest - F1: {rf_f1:.3f}, Accuracy: {rf_acc:.3f}")
    
    # XGBoost (best performance)
    xgb_binary = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42,
        eval_metric='logloss'
    )
    
    xgb_binary.fit(X_train_scaled, y_train_bin)
    xgb_pred = xgb_binary.predict(X_test_scaled)
    xgb_f1 = f1_score(y_test_bin, xgb_pred)
    xgb_acc = accuracy_score(y_test_bin, xgb_pred)
    
    print(f"XGBoost - F1: {xgb_f1:.3f}, Accuracy: {xgb_acc:.3f}")
    
    # Select best binary model
    best_binary_f1 = max(rf_f1, xgb_f1)
    best_binary_model = xgb_binary if xgb_f1 > rf_f1 else rf_binary
    best_binary_name = "XGBoost" if xgb_f1 > rf_f1 else "Random Forest"
    
    print(f"\nBest Binary Model: {best_binary_name} (F1: {best_binary_f1:.3f})")
else:
    print("No data for training!")

## Multi-class Classification (Leak Type Detection)

In [ ]:
# Train 2 best multi-class models
if df is not None:
    print("\n=== MULTI-CLASS CLASSIFICATION ===")
    
    # Random Forest
    rf_multi = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    
    rf_multi.fit(X_train_multi, y_train_multi)
    rf_pred_multi = rf_multi.predict(X_test_multi)
    rf_f1_w = f1_score(y_test_multi, rf_pred_multi, average='weighted')
    rf_f1_m = f1_score(y_test_multi, rf_pred_multi, average='macro')
    rf_acc = accuracy_score(y_test_multi, rf_pred_multi)
    
    print(f"Random Forest - F1(w): {rf_f1_w:.3f}, F1(m): {rf_f1_m:.3f}, Acc: {rf_acc:.3f}")
    
    # XGBoost
    xgb_multi = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42,
        eval_metric='mlogloss'
    )
    
    xgb_multi.fit(X_train_multi, y_train_multi)
    xgb_pred_multi = xgb_multi.predict(X_test_multi)
    xgb_f1_w = f1_score(y_test_multi, xgb_pred_multi, average='weighted')
    xgb_f1_m = f1_score(y_test_multi, xgb_pred_multi, average='macro')
    xgb_acc = accuracy_score(y_test_multi, xgb_pred_multi)
    
    print(f"XGBoost - F1(w): {xgb_f1_w:.3f}, F1(m): {xgb_f1_m:.3f}, Acc: {xgb_acc:.3f}")
    
    # Select best multi-class model
    best_multi_f1 = max(rf_f1_w, xgb_f1_w)
    best_multi_model = xgb_multi if xgb_f1_w > rf_f1_w else rf_multi
    best_multi_name = "XGBoost" if xgb_f1_w > rf_f1_w else "Random Forest"
    
    print(f"\nBest Multi-class Model: {best_multi_name} (F1-weighted: {best_multi_f1:.3f})")
    
    # Per-class performance
    classes = sorted(y_test_multi.unique())
    print("\nPer-class F1 scores:")
    for cls in classes:
        if best_multi_f1 == xgb_f1_w:
            cls_f1 = f1_score(y_test_multi == cls, xgb_pred_multi == cls)
        else:
            cls_f1 = f1_score(y_test_multi == cls, rf_pred_multi == cls)
        status = "EXCELLENT" if cls_f1 > 0.8 else "GOOD" if cls_f1 > 0.7 else "NEEDS WORK"
        print(f"  {cls:15s}: {cls_f1:.3f} ({status})")
else:
    print("No data for training!")

## Save Models for Real-time System

In [ ]:
# Save best models for deployment
if df is not None:
    # Create models directory
    models_dir = Path().absolute().parent / 'outputs' / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    
    # Save binary model
    joblib.dump(best_binary_model, models_dir / 'leak_binary.joblib')
    joblib.dump(scaler, models_dir / 'feature_scaler.pkl')
    
    # Save multi-class model
    joblib.dump(best_multi_model, models_dir / 'leak_multi.joblib')
    
    # Save feature names
    feature_names = X.columns.tolist()
    joblib.dump(feature_names, models_dir / 'feature_names.pkl')
    
    # Save metadata
    metadata = {
        'binary_model': {'type': best_binary_name, 'f1': best_binary_f1},
        'multiclass_model': {'type': best_multi_name, 'f1_weighted': best_multi_f1},
        'features': feature_names,
        'target_classes': sorted(y.unique()),
        'enhanced_features': len(available_enhanced)
    }
    joblib.dump(metadata, models_dir / 'model_metadata.pkl')
    
    print(f"\n=== MODELS SAVED ===")
    print(f"Binary: {best_binary_name} (F1: {best_binary_f1:.3f})")
    print(f"Multi-class: {best_multi_name} (F1: {best_multi_f1:.3f})")
    print(f"Enhanced features: {len(available_enhanced)}")
    print(f"\nReady for real-time leak detection system!")
    print(f"\nNext steps:")
    print(f"1. Build UI dashboard")
    print(f"2. Create real-time simulator")
    print(f"3. Deploy to production")
else:
    print("No models to save!")